# 08. Trees, heaps and tries

**Goal.** Use hierarchical structures to get ordered access, priority access and prefix search, and know when balance matters.

**You will be able to:**
1. Show that an unbalanced BST degrades to O(n) and an AVL tree stays at O(log n).
2. Implement and apply heaps: top-k, running median, schedulers and k-way merges.
3. Build a trie for autocomplete and compare its memory and speed with a sorted list.
4. Traverse trees iteratively, so deep inputs cannot overflow the call stack.

## 1. Binary search trees and balance

### How it works under the hood
- Each node holds a key and two child pointers. Everything in the left subtree is smaller, everything in the right subtree is larger.
- Search, insert and delete walk one root-to-leaf path: O(h), where h is the height.
- Inserting sorted keys into a plain BST builds a chain: h = n and every operation is O(n).
- **AVL trees** store a height per node and keep the left and right heights within 1. After an insert, a single or double **rotation** restores balance, so h stays near 1.44 log2(n).
- In-order traversal visits keys in sorted order. Recursion depth equals h, so use an explicit stack for untrusted or deep trees. CPython's default limit is 1000 frames.
- Databases and file systems use B-trees: the same idea with hundreds of keys per node, to cut disk reads.

### Production use
- Ordered maps and range queries: price ladders, scheduling intervals, leaderboards.
- Database indexes (B-tree and B+tree variants) and sorted key-value stores.
- Python has no built-in balanced tree. `sortedcontainers.SortedList` and `bisect` on a list fill the gap.

### Trade-offs

| Structure | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| Plain BST | Node with 2 pointers | O(log n) random, O(n) sorted input | Short code | Simple, ordered | Degenerates on sorted input |
| AVL tree | Node plus height | O(log n) always | Rotations take care | Guaranteed balance, fast lookups | More work per insert |
| Sorted list + `bisect` | Compact | O(log n) search, O(n) insert | Tiny code | Fast reads, cache friendly | Slow inserts on large lists |
| `dict` | Hash table | O(1) lookup | Direct | Fastest exact lookup | No ordering or range queries |

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "understanding_python").is_dir())
sys.path.insert(0, str(ROOT / "src"))
print("python", sys.version.split()[0], "| package root", ROOT.name)

python 3.13.6 | package root understanding-python


In [2]:
import random
import timeit

from understanding_python import trees as tr
from understanding_python.structures import AVLTree, BST

n = 2000
keys = list(range(n))
plain, avl, shuffled_bst = BST(), AVLTree(), BST()
for k in keys:
    plain.insert(k)
    avl.insert(k)
shuffle = keys[:]
random.Random(3).shuffle(shuffle)
for k in shuffle:
    shuffled_bst.insert(k)

print(f"{n} sorted inserts : plain BST height {plain.height():5d} | AVL height {avl.height():3d} (log2 n = {n.bit_length() - 1})")
print(f"{n} shuffled inserts: plain BST height {shuffled_bst.height():5d}")
print("AVL stays balanced:", avl.is_balanced(), "| in-order equals sorted keys:", avl.inorder() == keys)

probe = n - 1
t_chain = timeit.timeit(lambda: probe in plain, number=2000) / 2000
t_avl = timeit.timeit(lambda: probe in avl, number=2000) / 2000
print(f"lookup of the deepest key: chain {t_chain * 1e6:.1f} us | AVL {t_avl * 1e6:.2f} us")

root = tr.from_sorted(list(range(1, 16)))
print("level order of a perfectly balanced tree:", tr.level_order(root))
print("height:", tr.height(root), "| valid BST:", tr.is_valid_bst(root), "| LCA(1, 7):", tr.lowest_common_ancestor(root, 1, 7), "| LCA(9, 15):", tr.lowest_common_ancestor(root, 9, 15))

chain = None
for k in range(5000):
    chain = tr.insert_unbalanced(chain, k)
print("5000-node chain: height", tr.height(chain), "| iterative inorder ok:", tr.inorder(chain)[-1] == 4999)
root.left.key = 100
print("after corrupting a key, valid BST:", tr.is_valid_bst(root))

2000 sorted inserts : plain BST height  2000 | AVL height  11 (log2 n = 10)
2000 shuffled inserts: plain BST height    22
AVL stays balanced: True | in-order equals sorted keys: True


lookup of the deepest key: chain 177.8 us | AVL 1.06 us
level order of a perfectly balanced tree: [[8], [4, 12], [2, 6, 10, 14], [1, 3, 5, 7, 9, 11, 13, 15]]
height: 4 | valid BST: True | LCA(1, 7): 4 | LCA(9, 15): 12


5000-node chain: height 5000 | iterative inorder ok: True
after corrupting a key, valid BST: False


## 2. Heaps and priority queues

### How it works under the hood
- A binary heap is a complete binary tree stored in a flat list. The children of index `i` are `2i + 1` and `2i + 2`. The parent is `(i - 1) // 2`.
- The heap property: every parent is `<=` its children (min-heap). The smallest item sits at index 0.
- `push` appends and **sifts up**. `pop` moves the last item to the root and **sifts down**. Both are O(log n). `heapify` builds a heap from a list in O(n).
- `heapq` is a C min-heap over a plain list. For a max-heap, negate the keys or wrap items.
- Heaps do not support fast search or ordered iteration. They only guarantee the minimum.
- Ties compare the next tuple element. Add a counter as a tie-breaker, or you may compare unorderable objects.

### Production use
- Top-k queries over streams: trending products, largest files, slowest requests.
- Schedulers and event simulations; Dijkstra and A* frontiers.
- Merging sorted log shards or database runs (k-way merge).
- Running median for latency monitoring.

### Trade-offs

| Structure | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| Binary heap (`heapq`) | One list, no pointers | O(log n) push and pop, O(1) peek | Small API | Compact, very fast | No search, no decrease-key |
| Sorted list | Compact | O(n) insert, O(1) pop at the end | Obvious | Full order available | Slow updates |
| Full sort for top-k | Copy of all items | O(n log n) | One line | Simplest | Wasteful when k is small |
| Balanced tree | Node per item | O(log n) everything | More code | Ordered iteration, delete any | Heavier than a heap |

In [3]:
import heapq
import statistics

from understanding_python.bench import best_time
from understanding_python.heaps import RunningMedian, TaskScheduler, merge_sorted_streams, top_k
from understanding_python.structures import MinHeap

h = MinHeap()
for x in (9, 4, 7, 1, 8, 2):
    h.push(x)
print("array layout of the heap:", h.a, "| root is the minimum:", h.a[0])
print("pops in order:", [h.pop() for _ in range(len(h))])

rng = random.Random(21)
products = [(f"sku-{i:06d}", round(rng.random() * 500, 2)) for i in range(200_000)]
fast = top_k(products, 5)
slow = sorted(products, key=lambda p: -p[1])[:5]
print("top 5 by revenue:", fast[:3], "...")
print("same scores as a full sort:", [s for _, s in fast] == [s for _, s in slow], "(ties may order differently)")
t_heap = best_time(lambda: top_k(products, 5), repeat=2)
t_sort = best_time(lambda: sorted(products, key=lambda p: -p[1])[:5], repeat=2)
t_nl = best_time(lambda: heapq.nlargest(5, products, key=lambda p: p[1]), repeat=2)
print(f"200k items, k = 5: heap top_k {t_heap * 1e3:.0f} ms | heapq.nlargest {t_nl * 1e3:.0f} ms | full sort {t_sort * 1e3:.0f} ms")

latencies = [rng.randint(20, 400) for _ in range(5000)]
rm, window = RunningMedian(), []
for x in latencies:
    rm.add(x)
    window.append(x)
print("running median of 5000 latencies:", rm.median(), "| statistics.median:", statistics.median(window))

sched = TaskScheduler()
for name, prio in (("send-invoice", 2), ("reindex", 5), ("page-oncall", 0), ("send-receipt", 2), ("cleanup", 9)):
    sched.add(name, prio)
sched.cancel("cleanup")
sched.add("reindex", 1)
print("schedule order:", [sched.pop() for _ in range(len(sched))])

shards = [[1, 4, 9, 12], [2, 3, 10], [5, 6, 7, 8, 11]]
merged = list(merge_sorted_streams(*shards))
print("k-way merge:", merged, "| matches heapq.merge:", merged == list(heapq.merge(*shards)))

array layout of the heap: [1, 4, 2, 9, 8, 7] | root is the minimum: 1
pops in order: [1, 2, 4, 7, 8, 9]


top 5 by revenue: [('sku-170793', 500.0), ('sku-010628', 500.0), ('sku-174547', 499.99)] ...
same scores as a full sort: True (ties may order differently)


200k items, k = 5: heap top_k 22 ms | heapq.nlargest 16 ms | full sort 80 ms
running median of 5000 latencies: 208.0 | statistics.median: 208.0
schedule order: ['page-oncall', 'reindex', 'send-invoice', 'send-receipt']
k-way merge: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12] | matches heapq.merge: True


## 3. Tries

### How it works under the hood
- A trie stores one character per edge. A word is a path from the root, and a marker flags where a word ends.
- Lookup and insert cost O(L) for a word of length L, independent of how many words are stored.
- A prefix query walks to the prefix node in O(L), then collects the subtree. The cost is proportional to the matches, not the vocabulary.
- In Python each node is a `dict`. A dict costs hundreds of bytes even with few keys.
- A trie therefore often uses more memory than a sorted list of the same strings.
- Compressed forms (radix trees, DAWGs) merge single-child chains to cut that cost.

### Production use
- Autocomplete and search suggestions, spell checkers, IP routing tables (longest prefix match), URL routers.
- Dictionary games and tokenisers that match the longest known prefix.

### Trade-offs

| Structure | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| Trie of dicts | High in Python | O(L) lookup, O(L + matches) prefix | Clear | Prefix search independent of vocabulary size | Memory heavy, pointer chasing |
| Sorted list + `bisect` | Lowest | O(L log n) prefix range | Tiny code | Compact, cache friendly | Slow inserts, no per-character walk |
| `set` of strings | Medium | O(L) exact, O(n) prefix | Trivial | Fast exact match | Prefix needs a full scan |
| Radix tree or DAWG | Lowest of the tree forms | O(L) | Complex | Compact prefix structure | Harder to build and update |

In [4]:
import bisect

from understanding_python.internals import deep_size
from understanding_python.structures import Trie

ADJECTIVES = ["wireless", "waterproof", "compact", "organic", "ergonomic", "rechargeable", "foldable", "vintage"]
NOUNS = ["headphones", "backpack", "keyboard", "kettle", "lamp", "monitor", "mouse", "notebook", "speaker", "charger",
         "camera", "watch", "blender", "router", "drone", "bottle"]
COLOURS = ["black", "white", "blue", "red", "green", "grey"]
rng = random.Random(5)
terms = sorted({f"{rng.choice(ADJECTIVES)} {rng.choice(NOUNS)} {rng.choice(COLOURS)} {rng.randint(1, 999)}" for _ in range(40_000)})

trie = Trie()
for term in terms:
    trie.insert(term)

prefix = "wireless head"
via_trie = trie.with_prefix(prefix)
lo = bisect.bisect_left(terms, prefix)
hi = bisect.bisect_left(terms, prefix + "\uffff")
via_bisect = terms[lo:hi]
via_scan = [t for t in terms if t.startswith(prefix)]
print(f"{len(terms):,} distinct search terms | matches for {prefix!r}: {len(via_trie)}")
print("all three methods agree:", via_trie == via_bisect == via_scan)
print("first suggestions:", via_trie[:3])
print("exact search:", trie.search(terms[100]), "| missing term:", trie.search("wireless unicorn"))

t_trie = best_time(lambda: trie.with_prefix(prefix), repeat=3)
t_bis = best_time(lambda: terms[bisect.bisect_left(terms, prefix):bisect.bisect_left(terms, prefix + "\uffff")], repeat=3, number=10)
t_scan = best_time(lambda: [t for t in terms if t.startswith(prefix)], repeat=3)
print(f"prefix query: trie {t_trie * 1e6:.0f} us | bisect {t_bis * 1e6:.1f} us | linear scan {t_scan * 1e6:.0f} us")
print(f"memory: trie {deep_size(trie.root) / 1e6:.1f} MB vs sorted list {deep_size(terms) / 1e6:.1f} MB")

38,966 distinct search terms | matches for 'wireless head': 306


all three methods agree: True
first suggestions: ['wireless headphones black 129', 'wireless headphones black 166', 'wireless headphones black 188']
exact search: True | missing term: False
prefix query: trie 279 us | bisect 1.9 us | linear scan 1911 us
memory: trie 14.3 MB vs sorted list 2.9 MB
